In [1]:
import os
import time

import yaml

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from RLAlg.utils import set_seed_everywhere

from config import METAWORLD_CFGS, DMC_CFGS
from model.encoder import FrameObservationEncoderNet, EncoderNet
from state_frame_dataset import get_dataloader

In [ ]:
class Alignment(nn.Module):
    def __init__(self, vector_weight, config):
        super().__init__()

        self.state_encoder = EncoderNet(39, config["encoder_layers"])

        self.frame_encoder = FrameObservationEncoderNet(6, self.state_encoder.dim)


        self.state_encoder.load_state_dict(vector_weight)

        for param in self.state_encoder.parameters():
            param.requires_grad = False

    def encode_states(self, vectors):
        state_features = self.state_encoder.get_features(vectors)

        return state_features[-1]

    def encode_frames(self, frames):
        frame_features = self.frame_encoder(frames, False)

        return frame_features
    
    def forward(self, frames, states):
        frame_features = self.encode_frames(frames)
        state_features = self.encode_states(states)

        return frame_features, state_features

In [3]:
task_name = "pickplace"
seed = 0

In [4]:
if task_name in METAWORLD_CFGS:
    config_path = "configs/ddpg_metaworld.yaml"
elif task_name in DMC_CFGS:
    config_path = "configs/ddpg_dmc.yaml"

with open(config_path, "r") as file:
    config = yaml.safe_load(file)

weight_folder = f"weights/ddpg/{task_name}"
        
if not os.path.exists(weight_folder):
    os.makedirs(weight_folder)

set_seed_everywhere(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder_weight, _, _ = torch.load(f"{weight_folder}/actor_best_{seed}.pt", weights_only=True)
model = Alignment(encoder_weight, config).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=10)
dataloader = get_dataloader("pickplace")
size = len(dataloader.dataset)

In [5]:
def cosine_alignment_loss(x, y):
    #x = F.normalize(x, dim=-1)
    #y = F.normalize(y, dim=-1)
    #return 1 - (x * y).sum(dim=-1).mean()
    return F.mse_loss(x, y)

In [6]:
for _ in range(10):
    start_time = time.time()
    running_loss = 0.0
    for _, (vectors, frames) in enumerate(dataloader):
        vectors = vectors.to(device)
        frames = frames.to(device)
        
        frame_features, vector_features = model(frames, vectors)
        
        loss = cosine_alignment_loss(frame_features, vector_features)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * vectors.size(0)
    
    scheduler.step()
    end_time = time.time()
    train_loss = running_loss / size
    print(f"Train Loss: {train_loss:.4f}")
    elapsed_time = end_time - start_time
    print(f"The loop took {elapsed_time:.2f} seconds to complete.")

torch.save(model.frame_encoder.state_dict(), f"weights/ddpg/pickplace/frame_encoder_0_mse.pt")

TypeError: FrameObservationEncoderNet.forward() takes from 2 to 3 positional arguments but 4 were given

In [7]:
frame_features

tensor([[-0.1802, -0.1398,  1.2962,  ..., -0.1358,  0.9843,  0.5927],
        [ 0.5530,  1.7256, -0.3135,  ...,  0.0588,  0.1879,  1.1303],
        [ 0.2799,  0.3134,  0.5747,  ...,  0.6978,  0.0693,  0.0959],
        ...,
        [ 0.8666,  0.8154, -0.3169,  ...,  1.2750,  0.9815, -0.1639],
        [ 0.1450,  1.0962, -0.2252,  ..., -0.1477, -0.0359,  1.3437],
        [-0.2400, -0.0397,  1.0044,  ..., -0.1610,  0.1134,  0.3465]],
       device='cuda:0', grad_fn=<AddmmBackward0>)

In [8]:
vector_features

tensor([[-0.3294, -0.1992,  1.5377,  ..., -0.0999,  1.2700,  0.4809],
        [ 0.5228,  1.9958, -0.3482,  ...,  0.1878,  0.0518,  1.0667],
        [ 0.2749,  0.4840,  0.7082,  ...,  0.6926,  0.1227,  0.0484],
        ...,
        [ 0.9266,  0.6906, -0.2636,  ...,  1.3362,  0.9421, -0.1376],
        [ 0.0268,  1.0141, -0.2362,  ..., -0.1932,  0.0425,  1.3285],
        [-0.3574, -0.0105,  0.8118,  ..., -0.1555,  0.2844,  0.2853]],
       device='cuda:0')

In [9]:
F.cosine_similarity(frame_features, vector_features)

tensor([0.9885, 0.9795, 0.9801, 0.9161, 0.9868, 0.9877, 0.9914, 0.9787, 0.9717,
        0.9329, 0.9943, 0.9847, 0.9903, 0.9926, 0.9897, 0.9892],
       device='cuda:0', grad_fn=<SumBackward1>)